# Yerel Önbellekleme (Local caching)

Sunuculardan ağ üzerinden veri yüklemeyi öğrendiğinize göre, Flutter
uygulamanız artık daha canlı hissettirmelidir. Ancak, uzak sunuculardan
veri yükleyebiliyor olmanız, bunu her zaman yapmanız gerektiği anlamına
gelmez. Bazen, önceki ağ isteğinden aldığınız verileri tekrar etmek ve
kullanıcınızı tamamlanması için tekrar bekletmek yerine, bu verileri
yeniden oluşturmak (re-render) daha iyidir. Uygulama verilerini
gelecekte tekrar göstermek üzere saklama tekniğine **önbellekleme
(caching)** adı verilir ve bu sayfa, Flutter uygulamanızda bu göreve
nasıl yaklaşacağınızı kapsar.

## Önbelleklemeye giriş

En temel haliyle, tüm önbellekleme stratejileri, aşağıdaki sözde kodla
(pseudocode) temsil edilen aynı üç adımlı işleme dayanır:

``` dart
Data? _cachedData;

Future<Data> get data async {
    // Step 1: Check whether your cache already contains the desired data
    if (_cachedData == null) {
        // Step 2: Load the data if the cache was empty
        _cachedData = await _readData();
    }
    // Step 3: Return the value in the cache
    return _cachedData!;
}
```

Önbelleğin konumu, değerleri önbelleğe önceden yazma veya “ısıtma”
derecesi ve diğerleri dahil olmak üzere bu stratejiyi çeşitlendirmenin
birçok ilginç yolu vardır.

## Yaygın önbellekleme terminolojisi

Önbellekleme kendi terminolojisiyle birlikte gelir, bunlardan bazıları
aşağıda tanımlanmış ve açıklanmıştır.

### Önbellek isabeti (Cache hit)

Önbellek istenen bilgiyi zaten içerdiğinde ve gerçek doğruluk
kaynağından (source of truth) yüklenmesi gereksiz olduğunda, bir
uygulamanın bir **önbellek isabeti** yaşadığı söylenir.

### Önbellek ıskalaması (Cache miss)

Önbellek boş olduğunda ve istenen veriler gerçek doğruluk kaynağından
yüklendiğinde ve ardından gelecekteki okumalar için önbelleğe
kaydedildiğinde, bir uygulamanın bir **önbellek ıskalaması** yaşadığı
söylenir.

### Verileri önbelleklemenin riskleri

Doğruluk kaynağındaki veriler değiştiğinde, uygulamanın eski,
güncelliğini yitirmiş bilgileri oluşturma riski altında olduğu duruma
**bayat önbellek (stale cache)** denir.

Tüm önbellekleme stratejileri bayat verileri tutma riski taşır. Ne yazık
ki, bir önbelleğin tazeliğini doğrulama eylemi genellikle söz konusu
verileri tamamen yüklemek kadar zaman alır. Bu, çoğu uygulamanın
verileri önbelleklemekten yalnızca çalışma zamanında verilerin taze
olduğuna doğrulama yapmadan güvenirlerse yarar sağlama eğiliminde olduğu
anlamına gelir.

Bununla başa çıkmak için çoğu önbellekleme sistemi, önbelleğe alınmış
herhangi bir veri parçası üzerinde bir zaman sınırı içerir. Bu zaman
sınırı aşıldıktan sonra, taze veriler yüklenene kadar olası önbellek
isabetleri önbellek ıskalaması olarak ele alınır.

Bilgisayar bilimcileri arasında popüler bir şaka şudur: “Bilgisayar
bilimindeki en zor iki şey; önbellek geçersizleştirme (cache
invalidation), isimlendirme ve bir farkla hata (off-by-one errors)
yapmaktır.” 😄

Risklerine rağmen, dünyadaki hemen hemen her uygulama veri
önbelleklemesini yoğun bir şekilde kullanır. Bu sayfanın geri kalanı,
Flutter uygulamanızda verileri önbelleğe almak için birden fazla
yaklaşımı incelemektedir, ancak tüm bu yaklaşımların durumunuza göre
ayarlanabileceğini veya birleştirilebileceğini bilin.

## Yerel bellekte veri önbellekleme

En basit ve en performanslı önbellekleme stratejisi, bellek içi
(in-memory) önbellektir. Bu stratejinin dezavantajı, önbellek yalnızca
sistem belleğinde tutulduğu için, verilerin orijinal olarak önbelleğe
alındığı oturumun ötesinde tutulmamasıdır. (Elbette, bu “dezavantaj”
aynı zamanda çoğu bayat önbellek sorununu otomatik olarak çözme
avantajına da sahiptir!)

Basitlikleri nedeniyle, bellek içi önbellekler yukarıda görülen sözde
kodu yakından taklit eder. Bununla birlikte, kodunuzu düzenlemek ve
yukarıdaki gibi önbellek kontrollerinin tüm kod tabanınızda görünmesini
önlemek için **depo deseni (repository pattern)** gibi kanıtlanmış
tasarım ilkelerini kullanmak en iyisidir.

Yinelenen ağ isteklerini önlemek için kullanıcıları bellekte önbelleğe
almakla da görevli bir `UserRepository` sınıfı hayal edin. Uygulaması
şöyle görünebilir:

``` dart
class UserRepository {
  UserRepository(this.api);

  final Api api;
  final Map<int, User?> _userCache = {};

  Future<User?> loadUser(int id) async {
    if (!_userCache.containsKey(id)) {
      final response = await api.get(id);
      if (response.statusCode == 200) {
        _userCache[id] = User.fromJson(response.body);
      } else {
        _userCache[id] = null;
      }
    }
    return _userCache[id];
  }
}
```

Bu `UserRepository`, şunlar dahil olmak üzere birden fazla kanıtlanmış
tasarım ilkesini izler:

- Test etmeye yardımcı olan [bağımlılık enjeksiyonu (dependency
  injection)](https://en.wikipedia.org/wiki/Dependency_injection),
- Çevreleyen kodu uygulama ayrıntılarından koruyan [gevşek bağlılık
  (loose coupling)](https://en.wikipedia.org/wiki/Loose_coupling) ve
- Uygulamasının çok fazla endişeyle uğraşmasını önleyen [ilgi
  alanlarının ayrımı (separation of
  concerns)](https://en.wikipedia.org/wiki/Separation_of_concerns).

Ve en iyisi, bir kullanıcı tek bir oturumda belirli bir kullanıcıyı
yükleyen Flutter uygulamanızdaki sayfaları kaç kez ziyaret ederse etsin,
`UserRepository` sınıfı bu verileri ağ üzerinden yalnızca **bir kez**
yükler.

Ancak, kullanıcılarınız uygulamanızı her yeniden başlattıklarında
verilerin yüklenmesini beklemekten sonunda bıkabilirler. Bunun için,
aşağıda bulunan kalıcı önbellekleme stratejilerinden birini
seçmelisiniz.

## Kalıcı önbellekler (Persistent caches)

Verileri bellekte önbelleğe almak, değerli önbelleğinizin tek bir
kullanıcı oturumundan daha uzun süre yaşamasını asla sağlamaz.
Uygulamanızın yeni açılışlarında önbellek isabetlerinin performans
avantajlarından yararlanmak için, verileri cihazın sabit diskinde bir
yere önbelleğe almanız gerekir.

### `shared_preferences` ile veri önbellekleme

[`shared_preferences`](https://pub.dev/packages/shared_preferences),
Flutter’ın tüm altı hedef platformunda platforma özgü [anahtar-değer
depolamasını (key-value
storage)](https://en.wikipedia.org/wiki/Key%E2%80%93value_database)
saran bir Flutter eklentisidir. Bu temel platform anahtar-değer depoları
küçük veri boyutları için tasarlanmış olsa da, çoğu uygulama için bir
önbellekleme stratejisi olarak hala uygundur. Tam bir kılavuz için,
anahtar-değer depolarını kullanmayla ilgili diğer kaynaklarımıza bakın.

- [Diskte anahtar-değer verilerini
  saklama](https://docs.flutter.dev/cookbook/persistence/key-value)
- **Video:**:
  [`shared_preferences`](https://www.youtube.com/watch?v=sa_U0jffQII)

### Dosya sistemi ile veri önbellekleme

Flutter uygulamanız `shared_preferences` için ideal olan düşük işlem
hacimli senaryoları aşarsa, cihazınızın dosya sistemiyle veri önbelleğe
almayı keşfetmeye hazır olabilirsiniz. Daha kapsamlı bir kılavuz için,
dosya sistemi önbellekleme hakkındaki diğer kaynaklarımıza bakın.

- [Dosyaları okuma ve
  yazma](https://docs.flutter.dev/cookbook/persistence/reading-writing-files)

### Cihaz içi veritabanı ile veri önbellekleme

Yerel veri önbelleklemenin son aşaması, verileri okumak ve yazmak için
uygun bir veritabanı kullanan herhangi bir stratejidir. İlişkisel ve
ilişkisel olmayan veritabanları da dahil olmak üzere birden fazla tür
mevcuttur. Tüm yaklaşımlar, basit dosyalara göre - özellikle büyük veri
kümeleri için - önemli ölçüde iyileştirilmiş performans sunar. Daha
kapsamlı bir kılavuz için aşağıdaki kaynaklara bakın:

- [SQLite ile verileri kalıcı hale
  getirme](https://docs.flutter.dev/cookbook/persistence/sqlite)
- [SQLite alternatifi: `sqlite3`
  paketi](https://pub.dev/packages/sqlite3)
- [Drift, ilişkisel bir veritabanı: `drift`
  paketi](https://pub.dev/packages/drift)
- [Hive CE, ilişkisel olmayan bir veritabanı: `hive_ce`
  paketi](https://pub.dev/packages/hive_ce)
- [Isar Community, hızlı bir ilişkisel olmayan
  veritabanı:`isar_community`
  paketi](https://pub.dev/packages/isar_community)
- [Remote Caching, API yanıtları için hafif bir önbellekleme sistemi:
  `remote_caching` paketi](https://pub.dev/packages/remote_caching)

### Görüntüleri önbellekleme

Görüntüleri önbellekleme, normal verileri önbelleklemeye benzer bir
problem alanıdır, ancak herkese uyan tek bir çözümü vardır. Flutter
uygulamanızı görüntüleri saklamak için dosya sistemini kullanmaya
yönlendirmek üzere `cached_network_image` paketini kullanın.

- Video: [Haftanın Paketi:
  `cached_network_image`](https://www.youtube.com/watch?v=fnHr_rsQwDA)

## Durum geri yükleme (State restoration)

Uygulama verilerinin yanı sıra, kullanıcının oturumunun diğer yönlerini
de kalıcı hale getirmek isteyebilirsiniz; örneğin gezinme yığını
(navigation stack), kaydırma konumları ve hatta formları doldururken
kaydedilen kısmi ilerlemeler gibi. Bu desene “durum geri yükleme” denir
ve Flutter’da yerleşiktir.

Durum geri yükleme, Flutter framework’üne Element ağacındaki verileri
Flutter motoruyla senkronize etme talimatı vererek çalışır; motor daha
sonra bu verileri gelecek oturumlar için platforma özgü depolamada
önbelleğe alır. Android ve iOS için Flutter’da durum geri yüklemeyi
etkinleştirmek için aşağıdaki belgelere bakın:

- [**Android dokümantasyonu:** Android durum geri
  yükleme](https://docs.flutter.dev/platform-integration/android/restore-state-android)
- [**iOS dokümantasyonu:** iOS durum geri
  yükleme](https://docs.flutter.dev/platform-integration/ios/restore-state-ios)

## 📄 Lisans Bilgisi

Bu doküman, **Flutter resmi dokümantasyonundan** türetilmiş Türkçe ders
notudur.

**Orijinal kaynak:**  
https://docs.flutter.dev/get-started/fundamentals/networking

**Türkçe çeviri ve düzenleme:**  
[Doç. Dr. Hakan Temiz](mailto:htemiz@artvin.edu.tr)

------------------------------------------------------------------------

### Lisans Kapsamı

Bu dokümandaki içerikler aşağıdaki açık lisanslar kapsamında
sunulmaktadır:

**Metin içerikleri (anlatım ve açıklamalar):**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** Creative Commons Attribution 4.0 International (CC BY 4.0)  
https://creativecommons.org/licenses/by/4.0/

Bu lisans kapsamında: - İçerik kopyalanabilir, dağıtılabilir ve
uyarlanabilir  
- Ticari kullanım serbesttir  
- Kaynak belirtilmesi zorunludur

**Kod örnekleri:**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** BSD 3-Clause License  
https://opensource.org/licenses/BSD-3-Clause

Bu lisans kapsamında: - Kodlar kopyalanabilir, değiştirilebilir ve
dağıtılabilir  
- Ticari kullanım serbesttir  
- Lisans bildiriminin korunması gerekir